# Somatic frameshift and stop-gain by tissue of origin

In [ ]:
# 1) import modules
import os
import re
import json
import pandas as pd
import scipy
import time
import requests
import hashlib
import csv
import random
from collections import defaultdict
import numpy as np    
import statsmodels.api as sm   
import seaborn as sns
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

## Comparing between groups (tissue types and germline included)

In [465]:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)

somaticconcat_allTSG=pd.DataFrame()
alloutputdf=pd.DataFrame()

#3-all TSG, excluding splice region, all output stats to this df gene-wise
allTSG_df_nospliceregion=pd.DataFrame()

FLAG1_count=FLAG5_count=FLAGFISHER_count=0

cosmicconcat_OT_parent_annotated_readcsv=pd.read_csv("/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/COSMIC/Cosmic_MutantCensus_Tsv_v98_GRCh38/cosmicconcat_41TSGs_NCITtoOT_dropna_parentOT_11-25-24.txt", sep="\t")
        
parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)
# loading VEP consequence titles (category names) to use as index to join cbio and clinvar processed dfs:
VEP_calc_consequences=pd.read_csv("VEP_calculated_consequences_8-26-24.txt", sep="\t", header=None) #edited to add 'splice_site_variant' on 8-26-24


#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")

Hypermutators=pd.read_csv("cbio_sampleinfo_tmb_gt10_12-14-23.txt", sep="\t")

for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):# and (folder_name == "RB1"):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        
        clinvar_cbio_tocompare=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_cbio_tocompare_8-16-24.txt", sep="\t")
        Variation_interpretation_toexclude = ["Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic", "Conflicting interpretations of pathogenicity greater than or equal to 75"]
        clinvar_cbio_tocompare_VUSLBB=clinvar_cbio_tocompare[~clinvar_cbio_tocompare["GL_first_Description"].isin(Variation_interpretation_toexclude)]
        #print("shared count")
        #print(len(clinvar_cbio_tocompare))
        #print("shared that are VUSLBB")
        #print(len(clinvar_cbio_tocompare_VUSLBB))
        cbio_VEP=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_OTtissue_10-24-24.txt", sep="\t")
        #print("cbio VEP len")
        cbioCount1=len(cbio_VEP)
        #print(len(cbio_VEP))
        VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant"] #8-23-24 #, ">=50bp_indel"]
        cbio_VEP=cbio_VEP[~cbio_VEP["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        #print("cbio VEP len after filter VEP categories")
        #print(len(cbio_VEP))
        cbioCount2=len(cbio_VEP)
        Hypermutators_to_filter=Hypermutators.set_index('uniqueSampleKey').index
        #print("with hypermutators count QC same as above")
        totalbeforehypermutatorfilter=len(cbio_VEP)
        #print(len(cbio_VEP))
        cbio_VEP_nohypermutator=cbio_VEP[~cbio_VEP["uniqueSampleKey"].isin(Hypermutators_to_filter)]
        cbio_VEP=cbio_VEP_nohypermutator
        #print("without hypermutators count")
        #print(len(cbio_VEP))
        #print("% vars lost to hypermutator filter=#total-#afterfilter/#total")
        #print(((totalbeforehypermutatorfilter-len(cbio_VEP))*100)/totalbeforehypermutatorfilter)
        removetheseVUSLBB=clinvar_cbio_tocompare_VUSLBB["@id"]
        cbio_VEP_noVUSLBB=cbio_VEP[~cbio_VEP["@id"].isin(removetheseVUSLBB)]
        #print("without VUSLBB")
        #print(len(cbio_VEP_noVUSLBB))
        #print("# variants lost after VUSLBB filter")
        #print(len(cbio_VEP)-len(cbio_VEP_noVUSLBB))
        #print("% vars lost to VUSLBB filter=#total-#afterfilter/#total")
        #print(((len(cbio_VEP)-len(cbio_VEP_noVUSLBB))*100)/len(cbio_VEP))
        cbio_VEP_cancertypeall=cbio_VEP_noVUSLBB
        print(len(cbio_VEP_cancertypeall))
        #cbio=pd.concat([cbio, cbio_VEP_cancertype])
        cbio_VEP_cancertype_uniqueOT=cbio_VEP_cancertypeall[cbio_VEP_cancertypeall["unique_count_parents_in_OncoTree"]==1]
        print(len(cbio_VEP_cancertype_uniqueOT))
        
        cbio_VEP_cancertype_uniqueOT_0tummorAltCount=cbio_VEP_cancertype_uniqueOT[cbio_VEP_cancertype_uniqueOT["tumorAltCount"]==0]
        cbio_VEP_cancertype_uniqueOT_no0tummorAltCount=cbio_VEP_cancertype_uniqueOT.drop(cbio_VEP_cancertype_uniqueOT_0tummorAltCount.index)
        print(len(cbio_VEP_cancertype_uniqueOT_no0tummorAltCount))
        
        folder_name_aslist=[folder_name]
        cosmicconcat_OT_parent_annotated_readcsv_renamecolumns=cosmicconcat_OT_parent_annotated_readcsv.rename(columns={"unique_count_parents_in_OncoTree":'empty1', 'parent_tissue_code_list': 'empty2', "count_parents_in_OncoTree":"empty3", "parent_tissue_code":"empty4", '1': "unique_count_parents_in_OncoTree", '2': "parent_tissue_code_list", '3': "count_parents_in_OncoTree", '4':"parent_tissue_code"})

        cosmic=cosmicconcat_OT_parent_annotated_readcsv_renamecolumns[cosmicconcat_OT_parent_annotated_readcsv_renamecolumns["GENE_SYMBOL"].isin(folder_name_aslist)==True]
        print(len(cosmic))
        cosmic_uniqueOT=cosmic[cosmic["unique_count_parents_in_OncoTree"]==1]
        
        somaticconcat=pd.concat([cbio_VEP_cancertype_uniqueOT_no0tummorAltCount,cosmic_uniqueOT])
        print(len(somaticconcat))
        
        
        somaticconcat["patientid"]=somaticconcat["patientId"].astype("str")+somaticconcat["INDIVIDUAL_ID"].astype("str")
        somaticconcat=somaticconcat.rename(columns={"level_0":"level_0_hgvsg"})
        somaticconcat=somaticconcat.reset_index()
        
        somaticconcat_allTSG=pd.concat([somaticconcat_allTSG,somaticconcat])
        print(len(somaticconcat_allTSG))
        
        dfparentOT=pd.DataFrame(somaticconcat["parent_tissue_code"].value_counts())
        dfparentOT_filtergte25=dfparentOT[dfparentOT["count"]>=25]
        dfparentOT_filtergte25_resetindex=dfparentOT_filtergte25.reset_index()
        display(dfparentOT)
        adjpfactor=len(dfparentOT_filtergte25_resetindex)
        if (adjpfactor!=0):
            df_forstats=VEP_calc_consequences.set_index(0)
            
            for index, row in dfparentOT_filtergte25_resetindex.iterrows():
                OTparent=dfparentOT_filtergte25_resetindex.loc[index]["parent_tissue_code"]
                somaticconcat_singleOTparent=somaticconcat[somaticconcat["parent_tissue_code"].str.contains(OTparent.split("(")[0])==True]
                #print(len(somaticconcat_singleOTparent), OTparent)
                
                #initialize default dict to select 1 variant at random from list of variants per patient (for patients with multiple variants in the same gene):
                cbio_VEP_dict=defaultdict(list)
            
            
                # loading VEP consequence titles (categories) to use as index to join randomavg dfs:
                cbio_VEP_calc_consequences=VEP_calc_consequences
                cbio_VEP_randomavg_means=cbio_VEP_calc_consequences.set_index(0)
                #making dictionary of list of variants per patient id in the form ({patient1: [var1consequence, var2consequence, var3consequence,...], patient2:[var1consequence,var2consequence,..]})
                cbio_VEP_cancertype=somaticconcat_singleOTparent
                for index, row in cbio_VEP_cancertype.iterrows():
                    patientid=cbio_VEP_cancertype.loc[index]["patientid"]
                    VEPresult=cbio_VEP_cancertype.loc[index]["collapsed_Consequence"]
                    cbio_VEP_dict[patientid].append(VEPresult)

                #print(len(cbio_VEP_dict))
                #randomly selecting 1 variant per patient id and summarizing distribution, repeating 5 times
                for i in range(5):
                    cbio_VEP_random = {k:random.choice(v) for k,v in list(cbio_VEP_dict.items())}
                    list_VEP_random=[]
                    for key, val in cbio_VEP_random.items():
                        eachrow=[key,val]
                        list_VEP_random.append(eachrow)
                    #convert list to df of 1 variant per patient [patient1:var2consequence, patient2:var1consequence, ...]
                    dfconvert= pd.DataFrame(list_VEP_random)
                    #summarize counts per VEP category and convert to df having VEP category as index
                    cbio_VEP_randomavg=pd.DataFrame(dfconvert[1].value_counts())
                    #above is a df of current random selection having row header=VEP category and only 1 column of the counts per category (aka row) derived from current random selection
                    #rename counts column in above df with the loop number (1/2/3/4/5) + join together with previous df (if not the 1st random selection of 5) and calc mean of all 5 sets of distributions
                    rename=f"VEP_count_{i}"
                    cbio_VEP_randomavg_rename=cbio_VEP_randomavg.rename(columns={"count":rename})
                    cbio_VEP_randomavg_means=cbio_VEP_randomavg_means.join(cbio_VEP_randomavg_rename)
                #replace all missing values with 0 to calc mean
                cbio_VEP_randomavg_means_fill0=cbio_VEP_randomavg_means.fillna(0)
                #calc mean
                cbio_VEP_randomavg_means_fill0['mean']=cbio_VEP_randomavg_means_fill0.mean(axis=1)


                ## added 3-12-25:
                #dfparentOT_filtergte25_resetindex.groupby(by=["parent_tissue_code","collapsed_Consequence"]).size()

            
                #rename somatic column in cbio df
                cbio_VEP_randomavg_means_fill0_rename=cbio_VEP_randomavg_means_fill0.rename(columns={"mean":f"{OTparent}"})
                
                cbio_VEP_randomavg_means_fill0_rename=cbio_VEP_randomavg_means_fill0_rename.filter(items=[f"{OTparent}"])
                
                
                #combine gl and s dfs
                df_forstats = df_forstats.join(cbio_VEP_randomavg_means_fill0_rename)
                
            clinvar_VEP_cancertype=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_VEP_MANE_8-15-24.txt", sep="\t")
         
            clinvar_VEP_MANE=VEP_calc_consequences.set_index(0)
            #create list with each VEP category in data, plus occurrence list of number of times each variant seen in that category
            clinvar_VEPresult=defaultdict(list)
            for index, row in clinvar_VEP_cancertype.iterrows():
                VEPresult=clinvar_VEP_cancertype.loc[index]["collapsed_Consequence"]
                variantcount=clinvar_VEP_cancertype.loc[index]["Occurrence"]
                clinvar_VEPresult[VEPresult].append(variantcount)
            #sum the list of occurrence per VEP category
            clinvar_VEPresult_sumoccurrence = {k: sum(v) for (k, v) in clinvar_VEPresult.items()}
            #create dataframe from dictionary
            clinvar_dfconvert=pd.DataFrame.from_dict(clinvar_VEPresult_sumoccurrence, orient='index')
            #join with VEP consequence df
            clinvar_VEP=clinvar_VEP_MANE.join(clinvar_dfconvert)
            #replace missing values with 0
            clinvar_VEP_fill0=clinvar_VEP.fillna(0)
            #rename germline column
            clinvar_VEP_fill0_rename=clinvar_VEP_fill0.rename(columns={0:"VEP_germline"})
                
            df_gl_s = df_forstats.join(clinvar_VEP_fill0_rename)
                
            #filter out categories filtered by cbio (excep for TERT- diff set since cbio includes promoter variants)
            if folder_name=="TERT":
                VEP_categories_tofilter_list=["synonymous_variant", "3_prime_UTR_variant", "intron_variant", "downstream_gene_variant"]
            else:    
                VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant", "splice_donor_variant","splice_acceptor_variant", "missense_variant", "inframe_deletion", ">=50bp_indel", "inframe_insertion", "protein_altering_variant", "start_lost", "stop_lost"]

            df_gl_s_filter=df_gl_s.drop(VEP_categories_tofilter_list)
            #display(df_gl_s_filter)
            #df_gl_s_allgte5=df_gl_s_filter[(df_gl_s_filter.ne(0)).any(axis=1)]
            df_gl_s_allgte5=df_gl_s_filter[~(df_gl_s_filter < 5).all(axis=1)]
            display(df_gl_s_allgte5)
            print(df_gl_s_allgte5.sum())
             #run chi-square test from scipy
            from scipy.stats import chi2_contingency
            c_nospliceregion, p_nospliceregion, dof_nospliceregion, expected_nospliceregion = chi2_contingency(df_gl_s_allgte5)
            print(c_nospliceregion, p_nospliceregion, dof_nospliceregion) 
            display(df_gl_s_allgte5)
            print(expected_nospliceregion)

            expected_df=pd.DataFrame(expected_nospliceregion)
            count_less_than_5 = (expected_df < 5).sum().sum()
            if count_less_than_5/expected_df.size > 0.25:
                FLAG5=1
            else:
                FLAG5=0

            import numpy as np    
            import statsmodels.api as sm   
            table_nospliceregion = sm.stats.Table(df_gl_s_allgte5)   # Standardized residuals

               
            #Pearson_residuals to output
            Pearson_residuals_standardized_nospliceregion = table_nospliceregion.standardized_resids
            display(Pearson_residuals_standardized_nospliceregion)
            print(f"FLAG5 = {FLAG5}")

            outputdf=Pearson_residuals_standardized_nospliceregion
            alloutputdf=pd.concat([alloutputdf,outputdf])

        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)


Wed Mar 12 16:48:28 2025
131
125
125
471
596
596


/var/folders/84/djb4q50s33b6t8_cvx6h9src0000gn/T/ipykernel_2197/2812644074.py:16: DtypeWarning: Columns (24,177) have mixed types. Specify dtype option on import or set low_memory=False.
  cosmicconcat_OT_parent_annotated_readcsv=pd.read_csv("/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/COSMIC/Cosmic_MutantCensus_Tsv_v98_GRCh38/cosmicconcat_41TSGs_NCITtoOT_dropna_parentOT_11-25-24.txt", sep="\t")
/var/folders/84/djb4q50s33b6t8_cvx6h9src0000gn/T/ipykernel_2197/2812644074.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  somaticconcat["patientid"]=somaticconcat["patientId"].astype("str")+somaticconcat["INDIVIDUAL_ID"].astype("str")


,count
parent_tissue_code,
Myeloid (MYELOID),523
Lymphoid (LYMPH),14
Lung (LUNG),13
Esophagus/Stomach (STOMACH),9
Kidney (KIDNEY),7
Bowel (BOWEL),6
Skin (SKIN),6
Pancreas (PANCREAS),4
CNS/Brain (BRAIN),3


,Myeloid (MYELOID),VEP_germline
0,,
stop_gained,79.4,32.0
frameshift_variant,347.4,30.0


Myeloid (MYELOID)    426.8
VEP_germline          62.0
dtype: float64
31.672643519768272 1.8247455079090336e-08 1


...

Bowel (BOWEL)                          5863.0
Esophagus/Stomach (STOMACH)             182.0
Lung (LUNG)                             165.0
Prostate (PROSTATE)                      74.0
Liver (LIVER)                            43.0
Ampulla of Vater (AMPULLA_OF_VATER)      34.0
Biliary Tract (BILIARY_TRACT)            40.0
Skin (SKIN)                              34.0
Thyroid (THYROID)                        21.0
Pancreas (PANCREAS)                      25.0
VEP_germline                           2373.0
dtype: float64
347.45487889474697 1.3818181739041462e-68 10


,Bowel (BOWEL),Esophagus/Stomach (STOMACH),Lung (LUNG),Prostate (PROSTATE),Liver (LIVER),Ampulla of Vater (AMPULLA_OF_VATER),Biliary Tract (BILIARY_TRACT),Skin (SKIN),Thyroid (THYROID),Pancreas (PANCREAS),VEP_germline
0,,,,,,,,,,,
stop_gained,3421.8,88.0,83.4,29.4,16.2,14.8,20.2,27.4,7.8,10.6,866.0
frameshift_variant,2441.2,94.0,81.6,44.6,26.8,19.2,19.8,6.6,13.2,14.4,1507.0


[[3036.52279196   94.26013101   85.45561328   38.32554778   22.27025073
    17.60903546   20.71651231   17.60903546   10.87616896   12.94782019
  1229.00709284]
 [2826.47720804   87.73986899   79.54438672   35.67445222   20.72974927
    16.39096454   19.28348769   16.39096454   10.12383104   12.05217981
  1143.99290716]]


,Bowel (BOWEL),Esophagus/Stomach (STOMACH),Lung (LUNG),Prostate (PROSTATE),Liver (LIVER),Ampulla of Vater (AMPULLA_OF_VATER),Biliary Tract (BILIARY_TRACT),Skin (SKIN),Thyroid (THYROID),Pancreas (PANCREAS),VEP_germline
0,,,,,,,,,,,
stop_gained,17.325421,-0.938354,-0.323291,-2.085213,-1.857113,-0.965967,-0.163811,3.366901,-1.345009,-0.941061,-17.431063
frameshift_variant,-17.325421,0.938354,0.323291,2.085213,1.857113,0.965967,0.163811,-3.366901,1.345009,0.941061,17.431063


FLAG5 = 0
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/APC
Wed Mar 12 16:48:44 2025


## Comparing somatic combined with germline

In [593]:
## now run stats for all genes combining all somatic tissue types (gl vs s for frameshift vs stop-gain):

start_time_block2=time.asctime(time.localtime())
print(start_time_block2)

somaticconcat_allTSG=pd.DataFrame()
alloutputdf=pd.DataFrame()

#3-all TSG, excluding splice region, all output stats to this df gene-wise
allTSG_df_nospliceregion=pd.DataFrame()

FLAG1_count=FLAG5_count=FLAGFISHER_count=0

cosmicconcat_OT_parent_annotated_readcsv=pd.read_csv("/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/COSMIC/Cosmic_MutantCensus_Tsv_v98_GRCh38/cosmicconcat_41TSGs_NCITtoOT_dropna_parentOT_11-25-24.txt", sep="\t")
        
parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)
# loading VEP consequence titles (category names) to use as index to join cbio and clinvar processed dfs:
VEP_calc_consequences=pd.read_csv("VEP_calculated_consequences_8-26-24.txt", sep="\t", header=None) #edited to add 'splice_site_variant' on 8-26-24


#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")

Hypermutators=pd.read_csv("cbio_sampleinfo_tmb_gt10_12-14-23.txt", sep="\t")

for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):# and (folder_name == "RB1"):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        
        clinvar_cbio_tocompare=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_cbio_tocompare_8-16-24.txt", sep="\t")
        Variation_interpretation_toexclude = ["Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic", "Conflicting interpretations of pathogenicity greater than or equal to 75"]
        clinvar_cbio_tocompare_VUSLBB=clinvar_cbio_tocompare[~clinvar_cbio_tocompare["GL_first_Description"].isin(Variation_interpretation_toexclude)]
        #print("shared count")
        #print(len(clinvar_cbio_tocompare))
        #print("shared that are VUSLBB")
        #print(len(clinvar_cbio_tocompare_VUSLBB))
        cbio_VEP=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_OTtissue_10-24-24.txt", sep="\t")
        #print("cbio VEP len")
        cbioCount1=len(cbio_VEP)
        #print(len(cbio_VEP))
        VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant"] #8-23-24 #, ">=50bp_indel"]
        cbio_VEP=cbio_VEP[~cbio_VEP["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        #print("cbio VEP len after filter VEP categories")
        #print(len(cbio_VEP))
        cbioCount2=len(cbio_VEP)
        Hypermutators_to_filter=Hypermutators.set_index('uniqueSampleKey').index
        #print("with hypermutators count QC same as above")
        totalbeforehypermutatorfilter=len(cbio_VEP)
        #print(len(cbio_VEP))
        cbio_VEP_nohypermutator=cbio_VEP[~cbio_VEP["uniqueSampleKey"].isin(Hypermutators_to_filter)]
        cbio_VEP=cbio_VEP_nohypermutator
        #print("without hypermutators count")
        #print(len(cbio_VEP))
        #print("% vars lost to hypermutator filter=#total-#afterfilter/#total")
        #print(((totalbeforehypermutatorfilter-len(cbio_VEP))*100)/totalbeforehypermutatorfilter)
        removetheseVUSLBB=clinvar_cbio_tocompare_VUSLBB["@id"]
        cbio_VEP_noVUSLBB=cbio_VEP[~cbio_VEP["@id"].isin(removetheseVUSLBB)]
        #print("without VUSLBB")
        #print(len(cbio_VEP_noVUSLBB))
        #print("# variants lost after VUSLBB filter")
        #print(len(cbio_VEP)-len(cbio_VEP_noVUSLBB))
        #print("% vars lost to VUSLBB filter=#total-#afterfilter/#total")
        #print(((len(cbio_VEP)-len(cbio_VEP_noVUSLBB))*100)/len(cbio_VEP))
        cbio_VEP_cancertypeall=cbio_VEP_noVUSLBB
        print(len(cbio_VEP_cancertypeall))
        #cbio=pd.concat([cbio, cbio_VEP_cancertype])
        cbio_VEP_cancertype_uniqueOT=cbio_VEP_cancertypeall[cbio_VEP_cancertypeall["unique_count_parents_in_OncoTree"]==1]
        print(len(cbio_VEP_cancertype_uniqueOT))
        
        cbio_VEP_cancertype_uniqueOT_0tummorAltCount=cbio_VEP_cancertype_uniqueOT[cbio_VEP_cancertype_uniqueOT["tumorAltCount"]==0]
        cbio_VEP_cancertype_uniqueOT_no0tummorAltCount=cbio_VEP_cancertype_uniqueOT.drop(cbio_VEP_cancertype_uniqueOT_0tummorAltCount.index)
        print(len(cbio_VEP_cancertype_uniqueOT_no0tummorAltCount))
        
        folder_name_aslist=[folder_name]
        cosmicconcat_OT_parent_annotated_readcsv_renamecolumns=cosmicconcat_OT_parent_annotated_readcsv.rename(columns={"unique_count_parents_in_OncoTree":'empty1', 'parent_tissue_code_list': 'empty2', "count_parents_in_OncoTree":"empty3", "parent_tissue_code":"empty4", '1': "unique_count_parents_in_OncoTree", '2': "parent_tissue_code_list", '3': "count_parents_in_OncoTree", '4':"parent_tissue_code"})

        cosmic=cosmicconcat_OT_parent_annotated_readcsv_renamecolumns[cosmicconcat_OT_parent_annotated_readcsv_renamecolumns["GENE_SYMBOL"].isin(folder_name_aslist)==True]
        print(len(cosmic))
        cosmic_uniqueOT=cosmic[cosmic["unique_count_parents_in_OncoTree"]==1]
        
        somaticconcat=pd.concat([cbio_VEP_cancertype_uniqueOT_no0tummorAltCount,cosmic_uniqueOT])
        print(len(somaticconcat))
        
        
        somaticconcat["patientid"]=somaticconcat["patientId"].astype("str")+somaticconcat["INDIVIDUAL_ID"].astype("str")
        somaticconcat=somaticconcat.rename(columns={"level_0":"level_0_hgvsg"})
        somaticconcat=somaticconcat.reset_index()
        
        somaticconcat_allTSG=pd.concat([somaticconcat_allTSG,somaticconcat])
        print(len(somaticconcat_allTSG))
        
        #dfparentOT=pd.DataFrame(somaticconcat["parent_tissue_code"].value_counts())
        #dfparentOT_filtergte25=dfparentOT[dfparentOT["count"]>=25]
        #dfparentOT_filtergte25_resetindex=dfparentOT_filtergte25.reset_index()
        #display(dfparentOT)
        #adjpfactor=len(dfparentOT_filtergte25_resetindex)
        #if (adjpfactor!=0):
        #    df_forstats=VEP_calc_consequences.set_index(0)
        #    
        #    for index, row in dfparentOT_filtergte25_resetindex.iterrows():
        #       OTparent=dfparentOT_filtergte25_resetindex.loc[index]["parent_tissue_code"]
        #        somaticconcat_singleOTparent=somaticconcat[somaticconcat["parent_tissue_code"].str.contains(OTparent.split("(")[0])==True]
        #       #print(len(somaticconcat_singleOTparent), OTparent)
        #        
        #initialize default dict to select 1 variant at random from list of variants per patient (for patients with multiple variants in the same gene):
        cbio_VEP_dict=defaultdict(list)
            
            
        # loading VEP consequence titles (categories) to use as index to join randomavg dfs:
        cbio_VEP_calc_consequences=VEP_calc_consequences
        cbio_VEP_randomavg_means=cbio_VEP_calc_consequences.set_index(0)
        #making dictionary of list of variants per patient id in the form ({patient1: [var1consequence, var2consequence, var3consequence,...], patient2:[var1consequence,var2consequence,..]})
        #cbio_VEP_cancertype=somaticconcat_singleOTparent
        cbio_VEP_cancertype=somaticconcat
        for index, row in cbio_VEP_cancertype.iterrows():
            patientid=cbio_VEP_cancertype.loc[index]["patientid"]
            VEPresult=cbio_VEP_cancertype.loc[index]["collapsed_Consequence"]
            cbio_VEP_dict[patientid].append(VEPresult)

        #print(len(cbio_VEP_dict))
        #randomly selecting 1 variant per patient id and summarizing distribution, repeating 5 times
        for i in range(5):
            cbio_VEP_random = {k:random.choice(v) for k,v in list(cbio_VEP_dict.items())}
            list_VEP_random=[]
            for key, val in cbio_VEP_random.items():
                eachrow=[key,val]
                list_VEP_random.append(eachrow)
            #convert list to df of 1 variant per patient [patient1:var2consequence, patient2:var1consequence, ...]
            dfconvert= pd.DataFrame(list_VEP_random)
            #summarize counts per VEP category and convert to df having VEP category as index
            cbio_VEP_randomavg=pd.DataFrame(dfconvert[1].value_counts())
            #above is a df of current random selection having row header=VEP category and only 1 column of the counts per category (aka row) derived from current random selection
            #rename counts column in above df with the loop number (1/2/3/4/5) + join together with previous df (if not the 1st random selection of 5) and calc mean of all 5 sets of distributions
            rename=f"VEP_count_{i}"
            cbio_VEP_randomavg_rename=cbio_VEP_randomavg.rename(columns={"count":rename})
            cbio_VEP_randomavg_means=cbio_VEP_randomavg_means.join(cbio_VEP_randomavg_rename)
        #replace all missing values with 0 to calc mean
        cbio_VEP_randomavg_means_fill0=cbio_VEP_randomavg_means.fillna(0)
        #calc mean
        cbio_VEP_randomavg_means_fill0['mean']=cbio_VEP_randomavg_means_fill0.mean(axis=1)


        ## added 3-12-25:
        #dfparentOT_filtergte25_resetindex.groupby(by=["parent_tissue_code","collapsed_Consequence"]).size()

        OTparent="somatic"
        #rename somatic column in cbio df
        cbio_VEP_randomavg_means_fill0_rename=cbio_VEP_randomavg_means_fill0.rename(columns={"mean":f"{OTparent}"})
                
        cbio_VEP_randomavg_means_fill0_rename=cbio_VEP_randomavg_means_fill0_rename.filter(items=[f"{OTparent}"])
                
                
        #combine gl and s dfs
        #df_forstats = df_forstats.join(cbio_VEP_randomavg_means_fill0_rename)
                
        clinvar_VEP_cancertype=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_VEP_MANE_8-15-24.txt", sep="\t")
         
        clinvar_VEP_MANE=VEP_calc_consequences.set_index(0)
        #create list with each VEP category in data, plus occurrence list of number of times each variant seen in that category
        clinvar_VEPresult=defaultdict(list)
        for index, row in clinvar_VEP_cancertype.iterrows():
            VEPresult=clinvar_VEP_cancertype.loc[index]["collapsed_Consequence"]
            variantcount=clinvar_VEP_cancertype.loc[index]["Occurrence"]
            clinvar_VEPresult[VEPresult].append(variantcount)
        #sum the list of occurrence per VEP category
        clinvar_VEPresult_sumoccurrence = {k: sum(v) for (k, v) in clinvar_VEPresult.items()}
        #create dataframe from dictionary
        clinvar_dfconvert=pd.DataFrame.from_dict(clinvar_VEPresult_sumoccurrence, orient='index')
        #join with VEP consequence df
        clinvar_VEP=clinvar_VEP_MANE.join(clinvar_dfconvert)
        #replace missing values with 0
        clinvar_VEP_fill0=clinvar_VEP.fillna(0)
        #rename germline column
        clinvar_VEP_fill0_rename=clinvar_VEP_fill0.rename(columns={0:"VEP_germline"})
                
        df_gl_s = cbio_VEP_randomavg_means_fill0_rename.join(clinvar_VEP_fill0_rename)
                
        #filter out categories filtered by cbio (excep for TERT- diff set since cbio includes promoter variants)
        if folder_name=="TERT":
            VEP_categories_tofilter_list=["synonymous_variant", "3_prime_UTR_variant", "intron_variant", "downstream_gene_variant"]
        else:    
            VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant", "splice_donor_variant","splice_acceptor_variant", "missense_variant", "inframe_deletion", ">=50bp_indel", "inframe_insertion", "protein_altering_variant", "start_lost", "stop_lost"]

        df_gl_s_filter=df_gl_s.drop(VEP_categories_tofilter_list)
        #display(df_gl_s_filter)
        #df_gl_s_allgte5=df_gl_s_filter[(df_gl_s_filter.ne(0)).any(axis=1)]
        df_gl_s_allgte5=df_gl_s_filter[~(df_gl_s_filter < 5).all(axis=1)]
        display(df_gl_s_allgte5)
        print(df_gl_s_allgte5.sum())
        #run chi-square test from scipy
        from scipy.stats import chi2_contingency
        c_nospliceregion, p_nospliceregion, dof_nospliceregion, expected_nospliceregion = chi2_contingency(df_gl_s_allgte5)
        print(c_nospliceregion, p_nospliceregion, dof_nospliceregion) 
        display(df_gl_s_allgte5)
        print(expected_nospliceregion)

        expected_df=pd.DataFrame(expected_nospliceregion)
        count_less_than_5 = (expected_df < 5).sum().sum()
        if count_less_than_5/expected_df.size > 0.25:
            FLAG5=1
        else:
            FLAG5=0

        import numpy as np    
        import statsmodels.api as sm   
        table_nospliceregion = sm.stats.Table(df_gl_s_allgte5)   # Standardized residuals

               
        #Pearson_residuals to output
        Pearson_residuals_standardized_nospliceregion = table_nospliceregion.standardized_resids
        display(Pearson_residuals_standardized_nospliceregion)
        print(f"FLAG5 = {FLAG5}")

        outputdf=pd.DataFrame({"Gene":[folder_name],"p-value":[p_nospliceregion], "adjp":[p_nospliceregion*40],"somatic_ASPR":[Pearson_residuals_standardized_nospliceregion.loc["stop_gained"]["somatic"]], "germline_ASPR":[Pearson_residuals_standardized_nospliceregion.loc["stop_gained"]["VEP_germline"]]})
        alloutputdf=pd.concat([alloutputdf,outputdf])

        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)


Thu Jun 19 14:32:45 2025


/var/folders/84/djb4q50s33b6t8_cvx6h9src0000gn/T/ipykernel_73898/1991680147.py:16: DtypeWarning: Columns (24,177) have mixed types. Specify dtype option on import or set low_memory=False.
  cosmicconcat_OT_parent_annotated_readcsv=pd.read_csv("/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/COSMIC/Cosmic_MutantCensus_Tsv_v98_GRCh38/cosmicconcat_41TSGs_NCITtoOT_dropna_parentOT_11-25-24.txt", sep="\t")


131
125
125
471
596
596


/var/folders/84/djb4q50s33b6t8_cvx6h9src0000gn/T/ipykernel_73898/1991680147.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  somaticconcat["patientid"]=somaticconcat["patientId"].astype("str")+somaticconcat["INDIVIDUAL_ID"].astype("str")


,somatic,VEP_germline
0,,
stop_gained,113.4,32.0
frameshift_variant,362.6,30.0


...

somatic         6590.6
VEP_germline    2373.0
dtype: float64
301.582305921137 1.489509151622139e-67 1


,somatic,VEP_germline
0,,
stop_gained,3776.2,866.0
frameshift_variant,2814.4,1507.0


[[3413.23612388 1228.96387612]
 [3177.36387612 1144.03612388]]


,somatic,VEP_germline
0,,
stop_gained,17.390081,-17.390081
frameshift_variant,-17.390081,17.390081


FLAG5 = 0
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/APC
Thu Jun 19 14:33:03 2025
